In [1]:
!pip install pyspark

In [28]:
# Импорт необходимых библиотек
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from pyspark.sql import Row
import re

In [4]:
# Создание Spark сессии
spark = SparkSession.builder \
    .appName("Lab2_ProgrammingLanguagesReport") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark

In [5]:
# Загрузка данных постов
!wget https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/posts_sample.xml
postsData = spark.read.format('xml').option('rowTag', 'row').option("timestampFormat", 'y/M/d H:m:s').load('posts_sample.xml')

--2026-05-02 08:44:08--  https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/posts_sample.xml
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 74162295 (71M) [text/plain]
Saving to: ‘posts_sample.xml’

posts_sample.xml    100%[===================>]  70.73M   224KB/s    in 4m 16s  

2026-05-02 08:48:26 (282 KB/s) - ‘posts_sample.xml’ saved [74162295/74162295]



In [7]:
# Вывод информации о датасете постов
print("Структура данных:")
postsData.printSchema()

print("\nОбразец данных (первые 3 записи):")
postsData.show(n=3, truncate=30, vertical=True)

print("\nСтатистическая сводка:")
postsData.select("_Id", "_Score", "_ViewCount").summary().show()

print("\nОбщее количество записей в датасете:")
print(f"Всего постов: {postsData.count()}")

Структура данных:
root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: string (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: string (nullable = true)
 |-- _CreationDate: string (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: string (nullable = true)
 |-- _LastEditDate: string (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)


Образец данных (первые 3 записи):
-RECORD 0------------------------------

In [20]:
# Фильтрация постов за период 2010-2020 годы
dates = ("2010-01-01", "2020-12-31")
posts_by_date = postsData.filter(F.col("_CreationDate").between(*dates))

print(f"Постов за период 2010-2020: {posts_by_date.count()}")
print("\nПримеры постов из выбранного периода:")
posts_by_date.show(10)

Постов за период 2010-2020: 44419

Примеры постов из выбранного периода:
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|_Tags|_Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+---

In [11]:
# Загрузка данных языков программирования
!wget https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/programming-languages.csv
languagesData = spark.read.format('csv').option('header', 'true').option("inferSchema", True).load('programming-languages.csv').dropna()

--2026-05-02 08:52:13--  https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/programming-languages.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40269 (39K) [text/plain]
Saving to: ‘programming-languages.csv.1’

programming-languag 100%[===================>]  39.33K   150KB/s    in 0.3s    

2026-05-02 08:52:15 (150 KB/s) - ‘programming-languages.csv.1’ saved [40269/40269]



In [12]:
print("Описание структуры:")
languagesData.printSchema()

print("\nПервые 10 записей:")
languagesData.show(n=10, truncate=False)

print(f"\nКоличество языков в каталоге: {languagesData.count()}")

Описание структуры:
root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable = true)


Первые 10 записей:
+----------+---------------------------------------------------------+
|name      |wikipedia_url                                            |
+----------+---------------------------------------------------------+
|A# .NET   |https://en.wikipedia.org/wiki/A_Sharp_(.NET)             |
|A# (Axiom)|https://en.wikipedia.org/wiki/A_Sharp_(Axiom)            |
|A-0 System|https://en.wikipedia.org/wiki/A-0_System                 |
|A+        |https://en.wikipedia.org/wiki/A%2B_(programming_language)|
|A++       |https://en.wikipedia.org/wiki/A%2B%2B                    |
|ABAP      |https://en.wikipedia.org/wiki/ABAP                       |
|ABC       |https://en.wikipedia.org/wiki/ABC_(programming_language) |
|ABC ALGOL |https://en.wikipedia.org/wiki/ABC_ALGOL                  |
|ABSET     |https://en.wikipedia.org/wiki/ABSET                      |
|ABSYS     |https://e

In [16]:
# Получаем список названий языков
language_names = [str(row.name) for row in languagesData.collect()]
print(f"Всего загружено уникальных языков: {len(language_names)}")
for i, lang in enumerate(language_names, 1):
    print(f"{i:4}. {lang}")

Всего загружено уникальных языков: 699
   1. A# .NET
   2. A# (Axiom)
   3. A-0 System
   4. A+
   5. A++
   6. ABAP
   7. ABC
   8. ABC ALGOL
   9. ABSET
  10. ABSYS
  11. ACC
  12. Accent
  13. Ace DASL
  14. ACL2
  15. ACT-III
  16. Action!
  17. ActionScript
  18. Ada
  19. Adenine
  20. Agda
  21. Agilent VEE
  22. Agora
  23. AIMMS
  24. Alef
  25. ALF
  26. ALGOL 58
  27. ALGOL 60
  28. ALGOL 68
  29. ALGOL W
  30. Alice
  31. Alma-0
  32. AmbientTalk
  33. Amiga E
  34. AMOS
  35. AMPL
  36. Apex (Salesforce.com)
  37. APL
  38. App Inventor for Android's visual block language
  39. AppleScript
  40. Arc
  41. ARexx
  42. Argus
  43. AspectJ
  44. Assembly language
  45. ATS
  46. Ateji PX
  47. AutoHotkey
  48. Autocoder
  49. AutoIt
  50. AutoLISP / Visual LISP
  51. Averest
  52. AWK
  53. Axum
  54. B
  55. Babbage
  56. Bash
  57. BASIC
  58. bc
  59. BCPL
  60. BeanShell
  61. Batch (Windows/Dos)
  62. Bertrand
  63. BETA
  64. Bigwig
  65. Bistro
  66. BitC
  67. BLISS
 

In [31]:
# Функция определяет какой язык содержится в теге поста
def includes_name(x):
    tag = None
    for name in language_names:
        n = '<' + name.lower() + '>'
        if n in str(x._Tags).lower():
            tag = name
            break
    if tag is None:
        tag = 'No'

    # Извлекаем год из CreationDate
    year = int(x._CreationDate[:4])
    return (year, tag)

# Преобразование DataFrame posts_by_date в RDD, применение функции includes_name и фильтрация результатов
posts_by_date_rdd = posts_by_date.rdd.map(includes_name).filter(lambda x: x[1] != 'No')

# Группировка данных по году и языку программирования, подсчет количества записей
posts_by_date_rdd_group = posts_by_date_rdd.keyBy(lambda row: (row[0], row[1])) \
    .aggregateByKey(0, lambda x, y: x + 1, lambda x1, x2: x1 + x2)

# Получаем все доступные годы
available_years = sorted(set([row[0] for row in posts_by_date_rdd_group.keys().collect()]))

# Для каждого года создаём отдельную таблицу топ-10
print("Топ-10 языков программирования по годам")

for year in available_years:
    # Фильтруем данные для конкретного года
    year_data = posts_by_date_rdd_group.filter(lambda x: x[0][0] == year)

    # Сортируем по убыванию количества и берем топ-10
    top10 = year_data.sortBy(lambda x: x[1], ascending=False).take(10)

    # Создаём DataFrame для года
    row_template = Row('Year', 'Language', 'Count')
    year_df = spark.createDataFrame([row_template(year, lang, count) for (_, lang), count in top10])

    # Сохраняем как отдельную таблицу для года
    output_path = f"top10_languages/{year}"
    year_df.write.mode("overwrite").parquet(output_path)

    # Выводим результат
    print(f"\n{year} год:")
    year_df.show(truncate=False)

    # Также сохраняем в общий DataFrame для справки
    if year == available_years[0]:
        result_df = year_df
    else:
        result_df = result_df.union(year_df)

Топ-10 языков программирования по годам

2010 год:
+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2010|Java       |52   |
|2010|JavaScript |44   |
|2010|PHP        |42   |
|2010|Python     |25   |
|2010|Objective-C|23   |
|2010|C          |20   |
|2010|Ruby       |11   |
|2010|Delphi     |7    |
|2010|AppleScript|3    |
|2010|R          |3    |
+----+-----------+-----+


2011 год:
+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2011|PHP        |97   |
|2011|Java       |92   |
|2011|JavaScript |82   |
|2011|Python     |35   |
|2011|Objective-C|33   |
|2011|C          |24   |
|2011|Ruby       |17   |
|2011|Perl       |8    |
|2011|Delphi     |8    |
|2011|Bash       |7    |
+----+-----------+-----+


2012 год:
+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2012|PHP        |136  |
|2012|JavaScript |129  |
|2012|Java       |124  |
|2012|Python     |65   |
|2012|Objective-C|45   |
|2012|C          |27   |
